# Example 8.5 / 1: Find the Conflict in an Infeasible Steel Plan

Companion notebook for **8.5 Diagnosing Infeasibility with an IIS** in the AMPL lab Documentation, section "Find the Conflict in an Infeasible Steel Plan". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Find the Conflict in an Infeasible Steel Plan

In this version of the steel model, each product has a committed minimum, `commit`, used as a
lower bound on `Make`. The commitments need 41.8 hours but only 40 are available.
`option presolve 0;` keeps AMPL presolve from reporting the conflict itself, so the whole
problem reaches the solver. In `data.py` and `run.py`, `ampl` is an `AMPL` object that has
already read `model.mod`. First HiGHS, which returns a dual ray in the `.dunbdd` suffix:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
param commit {PROD} >= 0;                 # tons already promised to customers
var Make {p in PROD} >= commit[p], <= market[p];  # tons produced, within commitments and market
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["commit"] = {"bands": 3000, "coils": 2000, "plate": 2000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.option["presolve"] = 0
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print("Time dual ray:", ampl.get_value("Time.dunbdd"))

infeasible
Time dual ray: -1


### Another way in amplpy

HiGHS proves the model `infeasible`, and the nonzero dual-ray entry on `Time` points at the
hours constraint. A dual ray weighs rows, not variable bounds, so it cannot say which bounds take
part (Primal and Dual Rays (AMPL lab documentation, section 8.5.1)). Gurobi's `iisfind=1` asks for an IIS; the result comes
back in the symbolic suffix `.iis` on variables and constraints. With the same `model.mod` and
`data.py`:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["commit"] = {"bands": 3000, "coils": 2000, "plate": 2000}
ampl.param["avail"] = 40

In [8]:
# run.py
ampl.option["presolve"] = 0
ampl.solve(solver="gurobi", gurobi_options="iisfind=1", verbose=False)
print(ampl.solve_result)
print(ampl.get_data("Make.iis").to_pandas())
print("Time:", ampl.get_value("Time.iis"))

infeasible
      Make.iis
bands      low
coils      low
plate      low
Time: mem


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [9]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param commit := bands 3000  coils 2000  plate 2000;
param avail := 40;

Writing data.dat


In [10]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [11]:
%%ampl_eval
# model.run
option presolve 0;
option solver gurobi;
option gurobi_options 'iisfind=1';
option solver_msg 0;
solve;
display solve_result;
display Make.iis, Time.iis;

Gurobi 13.0.0:   iis:find = 1

suffix iis symbolic OUT;

option iis_table '\
0	non	not in the iis\
1	low	lower bound in the iis\
2	fix	both bounds in the iis\
3	upp	upper bound in the iis\
4	mem	member\
5	pmem	possible member\
6	plow	possibly lower bound\
7	pupp	possibly upper bound\
8	bug\
';
suffix dunbdd OUT;
solve_result = infeasible

Make.iis [*] :=
bands  low
coils  low
plate  low
;

Time.iis = mem

